In [19]:
tabla='hbeha10'
dim='dim_esthoshab'

In [20]:
import re

cadena = """
    esthabcod character(1) COLLATE pg_catalog."default",
    esthabdes character(30) COLLATE pg_catalog."default"
"""

# Reemplaza los caracteres innecesarios
cadena = cadena.replace(" COLLATE pg_catalog.\"default\",", "")
cadena = cadena.replace(" with time zone", "")

# Divide la cadena en una lista de líneas
lineas = cadena.split("\n")

# Crea la cadena de alteración de columnas
cadena_alter = ""
for i, linea in enumerate(lineas):
    palabras = linea.split()
    if len(palabras) >= 2:
        columna = palabras[0]
        esto = palabras[1]
        if i == len(lineas) - 2:
            # Última línea, agrega punto y coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {esto};\n"
        else:
            # Otras líneas, agrega coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {esto},\n"

# Utiliza una expresión regular para eliminar las comas duplicadas
cadena_alter = re.sub(r',+$', ',', cadena_alter, flags=re.MULTILINE)

print(cadena_alter)



import re

nombrecitos = re.findall(r'ALTER COLUMN\s+(\S+)', cadena_alter)
insertado_col = ','.join(nombrecitos)

print(insertado_col)

ALTER COLUMN esthabcod TYPE character(1),
ALTER COLUMN esthabdes TYPE character(30);

esthabcod,esthabdes


In [21]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
import decouple

config = decouple.AutoConfig(' ')

DB_USER=config("USER2_BDI_DW")
DB_PASSWORD=config("PASS2_BDI_DW")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_DW")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_DW")
DB_PASSWORD=config("PASS2_BDI_DW")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_DW")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [22]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527
engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()
base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection2)
connection2.close()

In [23]:
base2.head()

,esthabcod,esthabdes
0,1,NORMAL
1,2,EN CUARENTENA
2,3,PACIENTES INFECTO-CONTAGIOSOS


In [24]:
base2.columns

Index(['esthabcod', 'esthabdes'], dtype='object')

In [25]:
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")

Cantidad de filas en la tabla hbeha10 antes de la inserción: 3


In [26]:

base2.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

3

In [27]:
query=f"""
ALTER TABLE tmp_{tabla}

{cadena_alter}
UPDATE {tabla} b

SET

esthabdes = t.esthabdes

FROM tmp_{tabla} t 
WHERE 

b.esthabcod = t.esthabcod and b.esthabcod IS NOT NULL;

INSERT INTO {tabla}

({insertado_col})
SELECT 
{insertado_col}

FROM tmp_{tabla}  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {tabla} b 
    WHERE 
    
    b.esthabcod = t.esthabcod and b.esthabcod IS NOT NULL
) ;
"""

c1= text(query)
connection3.execute(c1)

In [28]:
#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
"""


c2= text(query2)
cursor=connection3.execute(c2)


In [29]:
query_count_after = f"SELECT COUNT(*) FROM {tabla}"
cant_despues = connection3.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla hbeha10 después de la inserción: 3
La cantidad de filas insertadas fue de: 0


In [30]:
base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection3)

In [31]:
#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

3

In [32]:
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

Cantidad de filas en la tabla dim_esthoshab antes de la inserción: 3


In [33]:
query=f"""

ALTER TABLE tmp_{tabla} 
{cadena_alter}
INSERT INTO {dim} 

(cod_esthab,des_esthab)

SELECT 

esthabcod,esthabdes

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d
    WHERE 
    
    d.cod_esthab = t.esthabcod

);
"""

c1= text(query)
cursor=connection4.execute(c1)


In [34]:
#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

In [35]:
c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_esthoshab después de la inserción: 3
La cantidad de filas insertadas fue de: 0


In [36]:
connection3.close()
connection4.close()